In [0]:

from pyspark.sql import functions as F
from datetime import datetime, timezone

quality_results = []

def log_check(table, check_name, failed_count, details=""):
    status = "PASS" if failed_count == 0 else "FAIL"

    quality_results.append({
        "table_name": table,
        "check_name": check_name,
        "failed_count": int(failed_count),
        "status": status,
        "details": details
    })

    print(f"{table} | {check_name} | {status} | {failed_count}")


In [0]:

# Check required Silver tables

required_tables = [
    "orders",
    "order_items",
    "order_reviews",
    "products",
    "customers",
    "sellers",
    "order_payments",
    "geolocation",
    "category_translation"
]

missing_tables = []

for name in required_tables:
    table_name = f"olist.silver.{name}"

    if spark.catalog.tableExists(table_name):
        print("FOUND:", table_name)
    else:
        print("MISSING:", table_name)
        missing_tables.append(table_name)

log_check(
    "olist.silver",
    "Required tables exist",
    len(missing_tables),
    ", ".join(missing_tables)
)


FOUND: olist.silver.orders
FOUND: olist.silver.order_items
MISSING: olist.silver.order_reviews
FOUND: olist.silver.products
FOUND: olist.silver.customers
FOUND: olist.silver.sellers
FOUND: olist.silver.order_payments
FOUND: olist.silver.geolocation
FOUND: olist.silver.category_translation
olist.silver | Required tables exist | FAIL | 1


In [0]:

# Check required Silver tables

required_tables = [
    "orders",
    "order_items",
    "order_reviews",
    "products",
    "customers",
    "sellers",
    "order_payments",
    "geolocation",
    "category_translation"
]

missing_tables = []

for name in required_tables:
    table_name = f"olist.silver.{name}"

    if spark.catalog.tableExists(table_name):
        print("FOUND:", table_name)
    else:
        print("MISSING:", table_name)
        missing_tables.append(table_name)

log_check(
    "olist.silver",
    "Required tables exist",
    len(missing_tables),
    ", ".join(missing_tables)
)


FOUND: olist.silver.orders
FOUND: olist.silver.order_items
MISSING: olist.silver.order_reviews
FOUND: olist.silver.products
FOUND: olist.silver.customers
FOUND: olist.silver.sellers
FOUND: olist.silver.order_payments
FOUND: olist.silver.geolocation
FOUND: olist.silver.category_translation
olist.silver | Required tables exist | FAIL | 1


In [0]:

from pyspark.sql import functions as F
from datetime import datetime, timezone

quality_results = []

def log_check(table, check_name, failed_count, details=""):
    status = "PASS" if failed_count == 0 else "FAIL"

    quality_results.append({
        "table_name": table,
        "check_name": check_name,
        "failed_count": int(failed_count),
        "status": status,
        "details": details
    })

    print(f"{table} | {check_name} | {status} | {failed_count}")


In [0]:

# Check required Silver tables

required_tables = [
    "orders",
    "order_items",
    "order_reviews",
    "products",
    "customers",
    "sellers",
    "order_payments",
    "geolocation",
    "category_translation"
]

missing_tables = []

for name in required_tables:
    table_name = f"olist.silver.{name}"

    if spark.catalog.tableExists(table_name):
        print("FOUND:", table_name)
    else:
        print("MISSING:", table_name)
        missing_tables.append(table_name)

log_check(
    "olist.silver",
    "Required tables exist",
    len(missing_tables),
    ", ".join(missing_tables)
)


FOUND: olist.silver.orders
FOUND: olist.silver.order_items
FOUND: olist.silver.order_reviews
FOUND: olist.silver.products
FOUND: olist.silver.customers
FOUND: olist.silver.sellers
FOUND: olist.silver.order_payments
FOUND: olist.silver.geolocation
FOUND: olist.silver.category_translation
olist.silver | Required tables exist | PASS | 0


In [0]:

# Compare record counts

row_counts = []

for name in required_tables:
    bronze_count = spark.table(
        f"olist.bronze.{name}"
    ).count()

    silver_count = spark.table(
        f"olist.silver.{name}"
    ).count()

    row_counts.append((
        name,
        bronze_count,
        silver_count,
        bronze_count - silver_count
    ))

counts_df = spark.createDataFrame(
    row_counts,
    [
        "table_name",
        "bronze_count",
        "silver_count",
        "difference"
    ]
)

display(counts_df)


table_name,bronze_count,silver_count,difference
orders,99441,99441,0
order_items,112650,112650,0
order_reviews,99224,98673,551
products,32951,32951,0
customers,99441,99441,0
sellers,3095,3095,0
order_payments,103886,103886,0
geolocation,1000163,738332,261831
category_translation,71,71,0


In [0]:

# Check missing business keys

primary_keys = {
    "orders": ["order_id"],
    "order_items": ["order_id", "order_item_id"],
    "order_reviews": ["order_id"],
    "products": ["product_id"],
    "customers": ["customer_id"],
    "sellers": ["seller_id"],
    "order_payments": ["order_id", "payment_sequential"],
    "geolocation": ["geolocation_zip_code_prefix"],
    "category_translation": ["product_category_name"]
}

for table, columns in primary_keys.items():

    df = spark.table(f"olist.silver.{table}")

    condition = None

    for c in columns:
        c_null = F.col(c).isNull()
        condition = c_null if condition is None else condition | c_null

    failed = df.filter(condition).count()

    log_check(
        table,
        "Missing business keys",
        failed
    )


orders | Missing business keys | PASS | 0
order_items | Missing business keys | PASS | 0
order_reviews | Missing business keys | PASS | 0
products | Missing business keys | PASS | 0
customers | Missing business keys | PASS | 0
sellers | Missing business keys | PASS | 0
order_payments | Missing business keys | PASS | 0
geolocation | Missing business keys | PASS | 0
category_translation | Missing business keys | PASS | 0


In [0]:

# Check duplicate business keys

unique_key_tables = {
    "orders": ["order_id"],
    "order_items": ["order_id", "order_item_id"],
    "order_reviews": ["order_id"],
    "products": ["product_id"],
    "customers": ["customer_id"],
    "sellers": ["seller_id"],
    "order_payments": ["order_id", "payment_sequential"],
    "category_translation": ["product_category_name"]
}

for table, keys in unique_key_tables.items():

    df = spark.table(f"olist.silver.{table}")

    duplicates = (
        df.groupBy(*keys)
        .count()
        .filter(F.col("count") > 1)
    )

    failed = duplicates.count()

    log_check(
        table,
        "Duplicate business keys",
        failed
    )


orders | Duplicate business keys | PASS | 0
order_items | Duplicate business keys | PASS | 0
order_reviews | Duplicate business keys | PASS | 0
products | Duplicate business keys | PASS | 0
customers | Duplicate business keys | PASS | 0
sellers | Duplicate business keys | PASS | 0
order_payments | Duplicate business keys | PASS | 0
category_translation | Duplicate business keys | PASS | 0


In [0]:

# Validate delivery chronology

orders = spark.table("olist.silver.orders")

invalid_dates = orders.filter(
    F.col("order_delivered_customer_date").isNotNull()
    & F.col("order_purchase_timestamp").isNotNull()
    & (
        F.col("order_delivered_customer_date")
        < F.col("order_purchase_timestamp")
    )
)

log_check(
    "orders",
    "Delivery before purchase",
    invalid_dates.count()
)

display(invalid_dates.limit(10))


orders | Delivery before purchase | PASS | 0


order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,ingestion_timestamp,source_file,delivery_delay_days,delivery_status


In [0]:

# Validate order-item amounts

items = spark.table("olist.silver.order_items")

invalid_items = items.filter(
    F.col("price").isNull()
    | (F.col("price") < 0)
    | F.col("freight_value").isNull()
    | (F.col("freight_value") < 0)
)

log_check(
    "order_items",
    "Invalid price or freight",
    invalid_items.count()
)


order_items | Invalid price or freight | PASS | 0


In [0]:

# Validate review scores

reviews = spark.table("olist.silver.order_reviews")

invalid_reviews = reviews.filter(
    F.col("review_score").isNull()
    | ~F.col("review_score").between(1, 5)
)

log_check(
    "order_reviews",
    "Invalid review scores",
    invalid_reviews.count()
)


order_reviews | Invalid review scores | PASS | 0


In [0]:

# Check order items without matching orders

orphan_items = items.join(
    orders.select("order_id"),
    on="order_id",
    how="left_anti"
)

log_check(
    "order_items",
    "Orders without matching parent",
    orphan_items.count()
)


order_items | Orders without matching parent | PASS | 0


In [0]:

# Check orders without matching customers

customers = spark.table("olist.silver.customers")

orphan_customers = orders.join(
    customers.select("customer_id"),
    on="customer_id",
    how="left_anti"
)

log_check(
    "orders",
    "Missing customer reference",
    orphan_customers.count()
)


orders | Missing customer reference | PASS | 0


In [0]:

products = spark.table("olist.silver.products")
sellers = spark.table("olist.silver.sellers")

missing_products = items.join(
    products.select("product_id"),
    on="product_id",
    how="left_anti"
).count()

missing_sellers = items.join(
    sellers.select("seller_id"),
    on="seller_id",
    how="left_anti"
).count()

log_check(
    "order_items",
    "Missing product reference",
    missing_products
)

log_check(
    "order_items",
    "Missing seller reference",
    missing_sellers
)


order_items | Missing product reference | PASS | 0
order_items | Missing seller reference | PASS | 0


In [0]:

# Create a quality results DataFrame

quality_df = spark.createDataFrame(
    quality_results
)

quality_df = quality_df.withColumn(
    "check_timestamp",
    F.current_timestamp()
)

# Save the results
(
    quality_df.write
    .format("delta")
    .mode("append")
    .saveAsTable("olist.silver.data_quality_log")
)

print("Data quality log saved successfully!")

display(quality_df)


Data quality log saved successfully!


check_name,details,failed_count,status,table_name,check_timestamp
Required tables exist,,0,PASS,olist.silver,2026-11-03T03:09:28.811Z
Missing business keys,,0,PASS,orders,2026-11-03T03:09:28.811Z
Missing business keys,,0,PASS,order_items,2026-11-03T03:09:28.811Z
Missing business keys,,0,PASS,order_reviews,2026-11-03T03:09:28.811Z
Missing business keys,,0,PASS,products,2026-11-03T03:09:28.811Z
Missing business keys,,0,PASS,customers,2026-11-03T03:09:28.811Z
Missing business keys,,0,PASS,sellers,2026-11-03T03:09:28.811Z
Missing business keys,,0,PASS,order_payments,2026-11-03T03:09:28.811Z
Missing business keys,,0,PASS,geolocation,2026-11-03T03:09:28.811Z
Missing business keys,,0,PASS,category_translation,2026-11-03T03:09:28.811Z


In [0]:

# Summarize current validation results

display(
    quality_df.groupBy("status")
    .agg(
        F.count("*").alias("total_checks")
    )
)


status,total_checks
PASS,25


In [0]:

# View failed checks, if any

display(
    quality_df.filter(
        F.col("status") == "FAIL"
    )
)


check_name,details,failed_count,status,table_name,check_timestamp


In [0]:

# Verify all nine Silver tables

required_tables = [
    "orders",
    "order_items",
    "order_reviews",
    "products",
    "customers",
    "sellers",
    "order_payments",
    "geolocation",
    "category_translation"
]

missing_tables = []

for table in required_tables:
    full_name = f"olist.silver.{table}"

    if spark.catalog.tableExists(full_name):
        print("FOUND:", full_name)
    else:
        print("MISSING:", full_name)
        missing_tables.append(full_name)

log_check(
    "olist.silver",
    "Required tables exist",
    len(missing_tables),
    ", ".join(missing_tables)
)


FOUND: olist.silver.orders
FOUND: olist.silver.order_items
FOUND: olist.silver.order_reviews
FOUND: olist.silver.products
FOUND: olist.silver.customers
FOUND: olist.silver.sellers
FOUND: olist.silver.order_payments
FOUND: olist.silver.geolocation
FOUND: olist.silver.category_translation
olist.silver | Required tables exist | PASS | 0


In [0]:

# Compare Bronze and Silver table counts

row_counts = []

for table in required_tables:

    bronze_count = spark.table(
        f"olist.bronze.{table}"
    ).count()

    silver_count = spark.table(
        f"olist.silver.{table}"
    ).count()

    row_counts.append((
        table,
        bronze_count,
        silver_count,
        bronze_count - silver_count
    ))

counts_df = spark.createDataFrame(
    row_counts,
    [
        "table_name",
        "bronze_count",
        "silver_count",
        "difference"
    ]
)

display(counts_df)


table_name,bronze_count,silver_count,difference
orders,99441,99441,0
order_items,112650,112650,0
order_reviews,99224,98673,551
products,32951,32951,0
customers,99441,99441,0
sellers,3095,3095,0
order_payments,103886,103886,0
geolocation,1000163,738332,261831
category_translation,71,71,0


In [0]:

# Check null values in important identifiers

keys = {
    "orders": ["order_id"],
    "order_items": ["order_id", "order_item_id"],
    "order_reviews": ["order_id"],
    "products": ["product_id"],
    "customers": ["customer_id"],
    "sellers": ["seller_id"],
    "order_payments": ["order_id", "payment_sequential"],
    "geolocation": ["geolocation_zip_code_prefix"],
    "category_translation": ["product_category_name"]
}

for table, columns in keys.items():

    df = spark.table(f"olist.silver.{table}")

    condition = None

    for column in columns:
        check = F.col(column).isNull()
        condition = check if condition is None else condition | check

    failed_count = df.filter(condition).count()

    log_check(
        table,
        "Missing business keys",
        failed_count
    )


orders | Missing business keys | PASS | 0
order_items | Missing business keys | PASS | 0
order_reviews | Missing business keys | PASS | 0
products | Missing business keys | PASS | 0
customers | Missing business keys | PASS | 0
sellers | Missing business keys | PASS | 0
order_payments | Missing business keys | PASS | 0
geolocation | Missing business keys | PASS | 0
category_translation | Missing business keys | PASS | 0


In [0]:

# Validate uniqueness of business keys

unique_keys = {
    "orders": ["order_id"],
    "order_items": ["order_id", "order_item_id"],
    "order_reviews": ["order_id"],
    "products": ["product_id"],
    "customers": ["customer_id"],
    "sellers": ["seller_id"],
    "order_payments": ["order_id", "payment_sequential"],
    "category_translation": ["product_category_name"]
}

for table, columns in unique_keys.items():

    df = spark.table(f"olist.silver.{table}")

    duplicates = (
        df.groupBy(*columns)
        .count()
        .filter(F.col("count") > 1)
    )

    log_check(
        table,
        "Duplicate business keys",
        duplicates.count()
    )


orders | Duplicate business keys | PASS | 0
order_items | Duplicate business keys | PASS | 0
order_reviews | Duplicate business keys | PASS | 0
products | Duplicate business keys | PASS | 0
customers | Duplicate business keys | PASS | 0
sellers | Duplicate business keys | PASS | 0
order_payments | Duplicate business keys | PASS | 0
category_translation | Duplicate business keys | PASS | 0


In [0]:

orders = spark.table("olist.silver.orders")
items = spark.table("olist.silver.order_items")
reviews = spark.table("olist.silver.order_reviews")

# Invalid delivery chronology
invalid_dates = orders.filter(
    F.col("order_delivered_customer_date").isNotNull()
    & F.col("order_purchase_timestamp").isNotNull()
    & (
        F.col("order_delivered_customer_date")
        < F.col("order_purchase_timestamp")
    )
).count()

log_check(
    "orders",
    "Delivery before purchase",
    invalid_dates
)

# Invalid item prices
invalid_prices = items.filter(
    F.col("price").isNull()
    | (F.col("price") < 0)
    | F.col("freight_value").isNull()
    | (F.col("freight_value") < 0)
).count()

log_check(
    "order_items",
    "Invalid price or freight",
    invalid_prices
)

# Invalid reviews
invalid_reviews = reviews.filter(
    F.col("review_score").isNull()
    | ~F.col("review_score").between(1, 5)
).count()

log_check(
    "order_reviews",
    "Invalid review scores",
    invalid_reviews
)


orders | Delivery before purchase | PASS | 0
order_items | Invalid price or freight | PASS | 0
order_reviews | Invalid review scores | PASS | 0


In [0]:

customers = spark.table("olist.silver.customers")
products = spark.table("olist.silver.products")
sellers = spark.table("olist.silver.sellers")

relationships = [
    ("order_items", items, "order_id", orders, "order_id"),
    ("orders", orders, "customer_id", customers, "customer_id"),
    ("order_items", items, "product_id", products, "product_id"),
    ("order_items", items, "seller_id", sellers, "seller_id")
]

for table, child, child_key, parent, parent_key in relationships:

    missing = child.join(
        parent.select(parent_key).distinct(),
        child[child_key] == parent[parent_key],
        "left_anti"
    ).count()

    log_check(
        table,
        f"Missing reference: {child_key}",
        missing
    )


order_items | Missing reference: order_id | PASS | 0
orders | Missing reference: customer_id | PASS | 0
order_items | Missing reference: product_id | PASS | 0
order_items | Missing reference: seller_id | PASS | 0


In [0]:

# Create and save the quality log

quality_df = spark.createDataFrame(quality_results)

quality_df = quality_df.withColumn(
    "check_timestamp",
    F.current_timestamp()
)

(
    quality_df.write
    .format("delta")
    .mode("append")
    .saveAsTable("olist.silver.data_quality_log")
)

print("Data quality log saved successfully!")

display(quality_df)


Data quality log saved successfully!


check_name,details,failed_count,status,table_name,check_timestamp
Required tables exist,,0,PASS,olist.silver,2026-11-03T03:12:35.556Z
Missing business keys,,0,PASS,orders,2026-11-03T03:12:35.556Z
Missing business keys,,0,PASS,order_items,2026-11-03T03:12:35.556Z
Missing business keys,,0,PASS,order_reviews,2026-11-03T03:12:35.556Z
Missing business keys,,0,PASS,products,2026-11-03T03:12:35.556Z
Missing business keys,,0,PASS,customers,2026-11-03T03:12:35.556Z
Missing business keys,,0,PASS,sellers,2026-11-03T03:12:35.556Z
Missing business keys,,0,PASS,order_payments,2026-11-03T03:12:35.556Z
Missing business keys,,0,PASS,geolocation,2026-11-03T03:12:35.556Z
Missing business keys,,0,PASS,category_translation,2026-11-03T03:12:35.556Z


In [0]:

display(
    quality_df.groupBy("status")
    .count()
)


status,count
PASS,50
